### Needs `01-Prep-PipeA.ipynb` to be executed first

Reads from:
- `PipelineA.json`
- `PipelineA_ynorm.json`

# Preperations

In [19]:
import pandas as pd
import numpy as np

## Import Data

In [2]:
# As this was the first evaluation script of this kind, I kept the non-ynom analysis for reasoning why we moved to ynorm as the standard
answers  = pd.read_json("PipelineA.json")
answers_ynorm = pd.read_json("PipelineA_ynorm.json")
CATEGORIES = ["value", "unit"]
VARIANTS   = ["b", "t", "ts"]   # bare | thinking | thinking_system

#### Cleanup Import

In [3]:
pipeline_cols = (
    [f"value_{v}" for v in VARIANTS] +
    [f"unit_{v}"  for v in VARIANTS] +
    [f"label_{v}" for v in VARIANTS]
)

for col in pipeline_cols:
    answers[col] = answers[col].apply(
    lambda x: np.nan if x is None else x
)
    
for col in pipeline_cols:
    answers_ynorm[col] = answers_ynorm[col].apply(
    lambda x: np.nan if x is None else x
)

## Defining Evaluation Functions

In [4]:
def check_hit(row, extraction_col, gs_col) -> bool:
    if isinstance(row[extraction_col], list): # Proceedes into nested code only if it is a list (and therefore has values in it)
        return row[gs_col] in row[extraction_col]
    else:
        return pd.isna(row[gs_col]) # Only executed if the extraction found nothing ("NaN") and returns True if there is really nothing in the Gold-Standard

def check_hit_exactly(row, extraction_col, gs_col) -> bool:
    # Preparing gold-standard value(s) for comparison
    gs_val  = row[gs_col]
    if isinstance(gs_val,  list):
        gs_set = set(gs_val)
    elif not pd.isna(gs_val):
        gs_set = {gs_val}
    else:
        gs_set = set()
    
    # Preparing extracted value(s) for comparison
    ext_val = row[extraction_col]
    if isinstance(ext_val, list):
        ext_set = set(ext_val)
    elif not pd.isna(ext_val):
        ext_set = {ext_val}
    else:
        ext_set = set()
    
    # Compare value(s), as there could be more than one (e.g., Allianz) or just one (mostly)
    return gs_set == ext_set


def eval_intern(source, result_basis, exact) -> pd.DataFrame:
    result = result_basis.copy()
    if exact:
        for cat in CATEGORIES: # "value" then "unit" comparison
            for v in VARIANTS: # "b" -> "t" -> "ts"
                result[f"{v}_{cat}_hit"] = source.apply(
                    check_hit_exactly,                  # Apply check for every row
                    args=(f"{cat}_{v}", cat),   # Forwarding arguments from for-loop
                    axis=1)                     # enables row-wise not column-wise applying (that would be axis=0)
    else:
        for cat in CATEGORIES: # "value" then "unit" comparison
            for v in VARIANTS: # "b" -> "t" -> "ts"
                result[f"{v}_{cat}_hit"] = source.apply(
                    check_hit,                  # Apply check for every row
                    args=(f"{cat}_{v}", cat),   # Forwarding arguments from for-loop
                    axis=1)                     # enables row-wise not column-wise applying (that would be axis=0)
                
    return result

def eval(source, exact) -> pd.DataFrame:
    
    # Creating smaller DataFrame to just see the True/False mapping
    gs_eval  = eval_intern(source, source[["report_name","year","scope"]], exact)
    
    # Second run, this time the returned DF is an extended form of the source DF (for further analysis)
    in_place = eval_intern(source, source, exact)
    
    return gs_eval, in_place



def print_eval(gs_eval) -> None: 
    for cat in CATEGORIES:
        print(f"### {cat} ###")
        for v in VARIANTS:
            col = gs_eval[f"{v}_{cat}_hit"]
            true_count  = col.value_counts()[True]
            false_count = col.value_counts()[False]
            quota       = round(col.mean()*100,2) #Percent
            print(f"{v:<2}: True = {true_count} || False = {false_count} || Quota = {quota}%")
        print()

# Evaluations

## Straight Forward

In [5]:
just_eval, expended = eval(answers, False)

print_eval(just_eval)

### value ###
b : True = 2044 || False = 164 || Quota = 92.57%
t : True = 2031 || False = 177 || Quota = 91.98%
ts: True = 2041 || False = 167 || Quota = 92.44%

### unit ###
b : True = 1918 || False = 290 || Quota = 86.87%
t : True = 1914 || False = 294 || Quota = 86.68%
ts: True = 1910 || False = 298 || Quota = 86.5%



### Exact

In [6]:
just_eval_exact, _ = eval(answers, True)

print_eval(just_eval_exact)

### value ###
b : True = 2033 || False = 175 || Quota = 92.07%
t : True = 2020 || False = 188 || Quota = 91.49%
ts: True = 2034 || False = 174 || Quota = 92.12%

### unit ###
b : True = 1918 || False = 290 || Quota = 86.87%
t : True = 1914 || False = 294 || Quota = 86.68%
ts: True = 1910 || False = 298 || Quota = 86.5%



## YNROM

In [7]:
ynorm, expended_ynorm = eval(answers_ynorm, False)

print_eval(ynorm)

### value ###
b : True = 2112 || False = 96 || Quota = 95.65%
t : True = 2097 || False = 111 || Quota = 94.97%
ts: True = 2108 || False = 100 || Quota = 95.47%

### unit ###
b : True = 1979 || False = 229 || Quota = 89.63%
t : True = 1970 || False = 238 || Quota = 89.22%
ts: True = 1967 || False = 241 || Quota = 89.09%



### Exact

In [8]:
ynorm_exact, forGEPA = eval(answers_ynorm, True)

print_eval(ynorm_exact)

### value ###
b : True = 2096 || False = 112 || Quota = 94.93%
t : True = 2082 || False = 126 || Quota = 94.29%
ts: True = 2100 || False = 108 || Quota = 95.11%

### unit ###
b : True = 1979 || False = 229 || Quota = 89.63%
t : True = 1970 || False = 238 || Quota = 89.22%
ts: True = 1967 || False = 241 || Quota = 89.09%



# Detailed Comparison

In [20]:
def eval_comparison(eval_raw, eval_ynorm):
    """Speichert beide Evaluationen strukturiert zum Vergleich"""
    results = []
    
    for cat in CATEGORIES:
        for v in VARIANTS:
            col_raw = eval_raw[f"{v}_{cat}_hit"]
            col_ynorm = eval_ynorm[f"{v}_{cat}_hit"]
            
            results.append({
                "variant": v,
                "category": cat,
                "raw_true": col_raw.value_counts()[True],
                "ynorm_true": col_ynorm.value_counts()[True],
                "raw_false": col_raw.value_counts()[False],
                "ynorm_false": col_ynorm.value_counts()[False],
                "abs_improvmnt": col_ynorm.value_counts()[True] - col_raw.value_counts()[True],
                "raw_quota": round(col_raw.mean() * 100, 2),
                "ynorm_quota": round(col_ynorm.mean() * 100, 2),
                "delta_quota": round((col_ynorm.mean() - col_raw.mean()) * 100, 2)
            })
    
    df_results = pd.DataFrame(results)
    #df_results.to_json(filepath, orient="records", indent=2)
    #df_results.to_csv(filepath.replace(".json", ".csv"), index=False)
    
    return df_results

# Speichern und anzeigen
comparison = eval_comparison(just_eval, ynorm)
print(comparison)


  variant category  raw_true  ynorm_true  raw_false  ynorm_false  \
0       b    value      2044        2112        164           96   
1       t    value      2031        2097        177          111   
2      ts    value      2041        2108        167          100   
3       b     unit      1918        1979        290          229   
4       t     unit      1914        1970        294          238   
5      ts     unit      1910        1967        298          241   

   abs_improvmnt  raw_quota  ynorm_quota  delta_quota  
0             68      92.57        95.65         3.08  
1             66      91.98        94.97         2.99  
2             67      92.44        95.47         3.03  
3             61      86.87        89.63         2.76  
4             56      86.68        89.22         2.54  
5             57      86.50        89.09         2.58  


In [10]:
comparison_exact = eval_comparison(just_eval_exact, ynorm_exact)
print(comparison_exact)

  variant category  raw_true  ynorm_true  raw_false  ynorm_false  \
0       b    value      2033        2096        175          112   
1       t    value      2020        2082        188          126   
2      ts    value      2034        2100        174          108   
3       b     unit      1918        1979        290          229   
4       t     unit      1914        1970        294          238   
5      ts     unit      1910        1967        298          241   

   abs_improvmnt  raw_quota  ynorm_quota  delta_quota  
0             63      92.07        94.93         2.85  
1             62      91.49        94.29         2.81  
2             66      92.12        95.11         2.99  
3             61      86.87        89.63         2.76  
4             56      86.68        89.22         2.54  
5             57      86.50        89.09         2.58  


In [11]:
# No merge necessary as the two DFs have the same index
comparison_exact["raw_true_any"] = comparison["raw_true"]
comparison_exact["fewer_hits_raw"] = comparison_exact["raw_true"] - comparison_exact["raw_true_any"]
comparison_exact["ynorm_true_any"] = comparison["ynorm_true"]
comparison_exact["fewer_hits_ynorm"] = comparison_exact["ynorm_true"] - comparison_exact["ynorm_true_any"]
comparison_exact[["variant","category","raw_true","ynorm_true","raw_true_any","fewer_hits_raw","ynorm_true_any","fewer_hits_ynorm"]]

,variant,category,raw_true,ynorm_true,raw_true_any,fewer_hits_raw,ynorm_true_any,fewer_hits_ynorm
0,b,value,2033,2096,2044,-11,2112,-16
1,t,value,2020,2082,2031,-11,2097,-15
2,ts,value,2034,2100,2041,-7,2108,-8
3,b,unit,1918,1979,1918,0,1979,0
4,t,unit,1914,1970,1914,0,1970,0
5,ts,unit,1910,1967,1910,0,1967,0


* You can see that the hits have dropped in general. Interestingly not all equal, not even for the same model.
* The drops are more evenly distributed for units than for values.


# Variant Comparison
Bare (`b`) vs. Thinking (`t`) vs. Thinking + System-Prompt (`ts`), all on the ynorm data.
## (EXACT)

In [12]:
_, variants_exact = eval(answers_ynorm, True) #Fresh Calculation

variants_exact = variants_exact[[
    'report_name', 'year', 'scope', 'value', 'unit', 'unit_normalized', 'label',
    'value_b', 'value_t', 'value_ts',
    'unit_b', 'unit_t', 'unit_ts',
    'label_b', 'label_t', 'label_ts',
    'b_value_hit', 't_value_hit', 'ts_value_hit',
    'b_unit_hit', 't_unit_hit', 'ts_unit_hit'
]]

### Where thinking fixes a miss of the bare run

In [13]:
think_improvement_exact = (variants_exact["b_value_hit"] == False) & (variants_exact["t_value_hit"] == True)
variants_exact.loc[think_improvement_exact, ["report_name", "year", "scope", "value", "value_b", "value_t"]]

,report_name,year,scope,value,value_b,value_t
40,acuity brands inc_2022_report,2021,3,NaN,[27651800.0],NaN
372,bekaert (d) sa_2022_report,2019,3,6045636.0,"[4874911.0, 72870.0, 352555.0, 31487.0, 27261....",[6045636.0]
374,bekaert (d) sa_2022_report,2021,3,6736962.0,"[5446554.0, 113767.0, 348402.0, 44627.0, 32713...",[6736962.0]
375,bekaert (d) sa_2022_report,2022,3,6187086.0,"[4762032.0, 133995.0, 325115.0, 77972.0, 37984...",[6187086.0]
472,cardinal energy ltd_2021_report,2019,2lb,277.0,NaN,[277.0]
473,cardinal energy ltd_2021_report,2020,2lb,259.0,NaN,[259.0]
474,cardinal energy ltd_2021_report,2021,2lb,228.0,NaN,[228.0]
492,cardinal energy ltd_2021_report,2019,3,NaN,[277.0],NaN
493,cardinal energy ltd_2021_report,2020,3,NaN,[259.0],NaN
494,cardinal energy ltd_2021_report,2021,3,NaN,[228.0],NaN


### Where the system-prompt fixes a miss of the thinking run

In [14]:
sysp_improvement_exact = (variants_exact["t_value_hit"] == False) & (variants_exact["ts_value_hit"] == True)
variants_exact.loc[sysp_improvement_exact, ["report_name", "year", "scope", "value", "value_t", "value_ts"]]

,report_name,year,scope,value,value_t,value_ts
97,aixtron_2020_report,2017,2lb,NaN,[5408.0],NaN
429,cabot corp_2018_report,2016,2lb,0.34,NaN,[0.34]
430,cabot corp_2018_report,2017,2lb,0.33,NaN,[0.33]
431,cabot corp_2018_report,2018,2lb,0.32,NaN,[0.32]
439,cabot corp_2018_report,2016,2mb,NaN,[0.34],NaN
440,cabot corp_2018_report,2017,2mb,NaN,[0.33],NaN
441,cabot corp_2018_report,2018,2mb,NaN,[0.32],NaN
782,georg fischer ag_2018_report,2017,1,325.00,"[325.0, 304.0]",[325.0]
789,georg fischer ag_2018_report,2014,2lb,264.00,NaN,[264.0]
790,georg fischer ag_2018_report,2015,2lb,264.00,NaN,[264.0]


## (ANY)

In [15]:
_, variants_any = eval(answers_ynorm, False) #Fresh Calculation

variants_any = variants_any[[
    'report_name', 'year', 'scope', 'value', 'unit', 'unit_normalized', 'label',
    'value_b', 'value_t', 'value_ts',
    'unit_b', 'unit_t', 'unit_ts',
    'label_b', 'label_t', 'label_ts',
    'b_value_hit', 't_value_hit', 'ts_value_hit',
    'b_unit_hit', 't_unit_hit', 'ts_unit_hit'
]]

In [16]:
think_improvement = (variants_any["b_value_hit"] == False) & (variants_any["t_value_hit"] == True)
variants_any.loc[think_improvement, ["report_name", "year", "scope", "value", "value_b", "value_t"]]

,report_name,year,scope,value,value_b,value_t
40,acuity brands inc_2022_report,2021,3,NaN,[27651800.0],NaN
472,cardinal energy ltd_2021_report,2019,2lb,277.0,NaN,[277.0]
473,cardinal energy ltd_2021_report,2020,2lb,259.0,NaN,[259.0]
474,cardinal energy ltd_2021_report,2021,2lb,228.0,NaN,[228.0]
492,cardinal energy ltd_2021_report,2019,3,NaN,[277.0],NaN
493,cardinal energy ltd_2021_report,2020,3,NaN,[259.0],NaN
494,cardinal energy ltd_2021_report,2021,3,NaN,[228.0],NaN
1207,inchcape plc_2022_report,2019,2mb,NaN,[50801.0],NaN
1209,inchcape plc_2022_report,2021,2mb,NaN,"[32949.0, 2486.0]",NaN
1248,innospec inc_2020_report,2020,2mb,0.0,NaN,[0.0]


In [17]:
sysp_improvement = (variants_any["t_value_hit"] == False) & (variants_any["ts_value_hit"] == True)
variants_any.loc[sysp_improvement, ["report_name", "year", "scope", "value", "value_t", "value_ts"]]

,report_name,year,scope,value,value_t,value_ts
97,aixtron_2020_report,2017,2lb,NaN,[5408.0],NaN
429,cabot corp_2018_report,2016,2lb,0.34,NaN,[0.34]
430,cabot corp_2018_report,2017,2lb,0.33,NaN,[0.33]
431,cabot corp_2018_report,2018,2lb,0.32,NaN,[0.32]
439,cabot corp_2018_report,2016,2mb,NaN,[0.34],NaN
440,cabot corp_2018_report,2017,2mb,NaN,[0.33],NaN
441,cabot corp_2018_report,2018,2mb,NaN,[0.32],NaN
789,georg fischer ag_2018_report,2014,2lb,264.00,NaN,[264.0]
790,georg fischer ag_2018_report,2015,2lb,264.00,NaN,[264.0]
791,georg fischer ag_2018_report,2016,2lb,299.00,NaN,[299.0]


#### See, which reports profit from "any" matching, so where multiple values got extracted and therefore "exact" did not match.

In [18]:
for v in VARIANTS:
    better_with_any = (variants_any[f"{v}_value_hit"] == True) & (variants_exact[f"{v}_value_hit"] == False)
    print(f"### {v} ###")
    display(variants_any.loc[better_with_any, ["report_name", "year", "scope", "value", f"value_{v}"]])

### b ###


,report_name,year,scope,value,value_b
345,bekaert (d) sa_2022_report,2022,1,316951.0,"[269969.0, 316951.0]"
372,bekaert (d) sa_2022_report,2019,3,6045636.0,"[4874911.0, 72870.0, 352555.0, 31487.0, 27261...."
374,bekaert (d) sa_2022_report,2021,3,6736962.0,"[5446554.0, 113767.0, 348402.0, 44627.0, 32713..."
375,bekaert (d) sa_2022_report,2022,3,6187086.0,"[4762032.0, 133995.0, 325115.0, 77972.0, 37984..."
1189,inchcape plc_2022_report,2021,1,9752.0,"[9752.0, 2486.0]"
1190,inchcape plc_2022_report,2022,1,17002.0,"[17002.0, 2791.0, 3617.0, 216.0]"
1199,inchcape plc_2022_report,2021,2lb,27277.0,"[27277.0, 3689.0]"
1200,inchcape plc_2022_report,2022,2lb,22223.0,"[22223.0, 2886.0]"
1219,inchcape plc_2022_report,2021,3,82068.0,"[82068.0, 9221.0]"
1220,inchcape plc_2022_report,2022,3,187713.0,"[187713.0, 2975.0]"


### t ###


,report_name,year,scope,value,value_t
99,aixtron_2020_report,2019,2lb,406.60,"[381.0, 171.74, 406.6]"
100,aixtron_2020_report,2020,2lb,398.14,"[557.0, 129.91, 398.14]"
101,aixtron_2020_report,2020,2lb,398.14,"[557.0, 129.91, 398.14]"
345,bekaert (d) sa_2022_report,2022,1,316951.00,"[269969.0, 316951.0]"
782,georg fischer ag_2018_report,2017,1,325.00,"[325.0, 304.0]"
1189,inchcape plc_2022_report,2021,1,9752.00,"[9752.0, 2486.0]"
1190,inchcape plc_2022_report,2022,1,17002.00,"[17002.0, 2791.0, 3617.0, 216.0]"
1199,inchcape plc_2022_report,2021,2lb,27277.00,"[27277.0, 3689.0]"
1200,inchcape plc_2022_report,2022,2lb,22223.00,"[22223.0, 2886.0]"
1219,inchcape plc_2022_report,2021,3,82068.00,"[82068.0, 9221.0]"


### ts ###


,report_name,year,scope,value,value_ts
99,aixtron_2020_report,2019,2lb,406.60,"[381.0, 171.74, 406.6]"
100,aixtron_2020_report,2020,2lb,398.14,"[557.0, 129.91, 398.14]"
101,aixtron_2020_report,2020,2lb,398.14,"[557.0, 129.91, 398.14]"
345,bekaert (d) sa_2022_report,2022,1,316951.00,"[269969.0, 316951.0]"
1190,inchcape plc_2022_report,2022,1,17002.00,"[17002.0, 2791.0]"
1265,jetblue airways corp_2019_report,2016,1,7484799.00,"[7491616.0, 7484799.0]"
1266,jetblue airways corp_2019_report,2016,1,7491616.00,"[7491616.0, 7484799.0]"
1969,sumitomo corporation_2021_report,2020,1,1526724.00,"[1526724.0, 1522514.0, 4210.0]"
